# Exploracao e Treino: Previsao de Rating de Problemas (XGBoost vs PyTorch MLP)

Notebook simples para analise exploratoria (EDA) e comparacao via `KFold` entre **XGBoost Regressor** e **PyTorch MLP** na previsao de dificuldade (800-2800) de problemas de programacao competitiva.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve() if Path("../src").exists() else Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt

from src.dataset import load_dataset
from src.features import extract_numeric_features
from src.train import train_and_evaluate

## 1. Analise Exploratoria (EDA)

In [ ]:
data_path = PROJECT_ROOT / "data" / "problems_sample.csv"
df = load_dataset(csv_path=data_path, generate_if_missing=True, n_samples=300, seed=42)
print("Dataset:", df.shape)
display(df.head())

num_feats = extract_numeric_features(df)
print("\nCorrelacao das features extraidas com Rating:")
print(num_feats.corrwith(df["rating"]).sort_values(ascending=False).round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(df["rating"], bins=15, color="#1f77b4", edgecolor="white")
axes[0].set_title("Distribuicao de Rating (800 - 2800)")
axes[0].set_xlabel("Rating")
axes[0].set_ylabel("Quantidade")

axes[1].scatter(num_feats["max_n_log10"], df["rating"], alpha=0.45, color="#ff7f0e")
axes[1].set_title("log10(max N) vs Rating")
axes[1].set_xlabel("log10(max N)")
axes[1].set_ylabel("Rating")
plt.tight_layout()
plt.show()

## 2. Treino e Comparacao (KFold 5-Fold)

In [ ]:
summary = train_and_evaluate(
    data_path=data_path,
    assets_dir=PROJECT_ROOT / "assets",
    artifacts_dir=PROJECT_ROOT / "artifacts",
    n_splits=5,
    epochs=35,
    seed=42,
)
print("XGBoost:    ", summary.xgb_metrics)
print("PyTorch MLP:", summary.mlp_metrics)